# 01 — Perfilado inicial (Entrega 1)

Objetivo: verificar volúmenes, tipos, nulos, cardinalidad e integridad referencial de las dos fuentes del proyecto. Las cifras de este notebook alimentan `docs/data_dictionary.md`.

El perfilado **formal y completo** (con reglas de limpieza y homologación) corresponde a la Entrega 2.

In [1]:
import pandas as pd

catalog = pd.read_csv('../data/raw/catalog_raw.csv')
movements = pd.read_csv('../data/raw/movements_raw.csv', parse_dates=['timestamp'])

print(f'Catálogo:    {catalog.shape[0]} filas × {catalog.shape[1]} columnas')
print(f'Movimientos: {movements.shape[0]} filas × {movements.shape[1]} columnas')
print(f'Rango temporal: {movements["timestamp"].min().date()} → {movements["timestamp"].max().date()}')
print(f'Hogares: {movements["household_id"].nunique()}')

Catálogo:    50 filas × 7 columnas
Movimientos: 25819 filas × 10 columnas
Rango temporal: 2026-02-01 → 2026-05-01
Hogares: 10


## Catálogo — tipos, nulos y cardinalidad

In [2]:
pd.DataFrame({
    'dtype': catalog.dtypes.astype(str),
    'nulos': catalog.isnull().sum(),
    'cardinalidad': catalog.nunique(),
})

,dtype,nulos,cardinalidad
proteins_100g,float64,0,34
carbs_100g,float64,0,40
calories_100g,float64,7,30
product_id,int64,0,50
product_name,object,0,50
category,int64,0,6
nutriscore,object,0,5


In [3]:
# Distribución de nutriscore y categorías
print('Nutriscore:', catalog['nutriscore'].value_counts().to_dict())
print(f"Categorías distintas: {catalog['category'].nunique()}")

Nutriscore: {'A': 20, 'B': 13, 'D': 7, 'Falta Dato': 7, 'C': 3}
Categorías distintas: 6


In [4]:
# Detección del outlier en calories_100g (R2 de la propuesta)
outliers = catalog[catalog['calories_100g'] > 900]
outliers[['product_id', 'product_name', 'calories_100g']]

,product_id,product_name,calories_100g


## Movimientos — tipos, nulos y cardinalidad

In [5]:
pd.DataFrame({
    'dtype': movements.dtypes.astype(str),
    'nulos': movements.isnull().sum(),
    'cardinalidad': movements.nunique(),
})

,dtype,nulos,cardinalidad
event_id,object,0,25819
household_id,int64,0,10
stock_id,object,0,11581
product_id,int64,0,50
product_name,object,0,50
event_type,object,0,2
quantity,int64,0,3
timestamp,datetime64[ns],0,19624
expiry_date,object,0,113
classification,object,0,4


In [6]:
print('Rango temporal:', movements['timestamp'].min(), '→', movements['timestamp'].max())
print('Hogares distintos:', movements['household_id'].nunique())
print('event_type:', movements['event_type'].value_counts().to_dict())
print('classification:', movements['classification'].value_counts().to_dict())

Rango temporal: 2026-02-01 10:01:36 → 2026-05-01 23:59:59
Hogares distintos: 10
event_type: {'OUT': 14238, 'IN': 11581}
classification: {'Purchase': 11581, 'Consumption': 9185, 'Forced_Waste': 3531, 'Waste': 1522}


## Integridad referencial

In [7]:
huerfanos = (~movements['product_id'].astype(str).isin(catalog['product_id'].astype(str))).sum()
print(f'Eventos con product_id ausente en el catálogo: {huerfanos}')
print(f'Integridad referencial: {100 - huerfanos/len(movements)*100:.1f}%')
print(f'(Catálogo ahora usa IDs Instacart alineados con la simulación)')

Eventos con product_id ausente en el catálogo: 0
Integridad referencial: 100.0%
(Catálogo ahora usa IDs Instacart alineados con la simulación)


## Nulos estructurales en `expiry_date`

En el esquema anterior (`action_type` / `location`) `expiry_date` era nulo en eventos `OUT`.  
En el nuevo esquema (simulación Instacart, 10 hogares) **todos los eventos registran `expiry_date`**, ya que el item ya estaba en inventario con su fecha de vencimiento conocida.

In [8]:
# expiry_date existe en todos los eventos (IN y OUT), no es un nulo estructural en el nuevo esquema
print('Nulos en expiry_date por event_type:')
print(movements.groupby('event_type')['expiry_date'].apply(lambda s: s.isnull().sum()))

print('\nNota: en el nuevo esquema ambos IN y OUT registran expiry_date.')
print('Los nulos estructurales del esquema anterior (action_type/location) ya no aplican.')

Nulos en expiry_date por event_type:
event_type
IN     0
OUT    0
Name: expiry_date, dtype: int64

Nota: en el nuevo esquema ambos IN y OUT registran expiry_date.
Los nulos estructurales del esquema anterior (action_type/location) ya no aplican.
